#THỰC HÀNH: CÁC GIẢI THUẬT PHÂN LOẠI CƠ BẢN

##GIẢI THUẬT 3: BAYES NGÂY THƠ (NAÏVE BAYES)
###Bài tập thực hành 2: Xây dựng mô hình Naïve ngây thơ trên tập dữ liệu mushroom.

###Bộ phân loại Naive Bayes từ đầu (Naive Bayes Classifier from Scratch)

Trong notebook này, chúng ta sẽ tìm hiểu tổng quan về thuật toán phân loại Naive Bayes, xây dựng nó từ đầu (tức là không dùng thư viện có sẵn), và thử áp dụng nó vào bộ dữ liệu nấm (mushroom dataset) để dự đoán xem một cây nấm có độc hay có thể ăn được.

Mô hình xác suất cho bộ phân loại này có dạng:


\begin{equation}
P(C_k | x) = \frac{P(C_k) * P(x | C_k)}{P(x)}
\end{equation}


Nói một cách đơn giản, theo thuật ngữ xác suất Bayes, công thức trên có thể được viết lại thành:

\begin{equation}
posterior = \frac{prior * likelihood}{evidence}
\end{equation}

Trong đó:
- **posterior**: xác suất sau khi quan sát dữ liệu
- **prior**: xác suất tiên nghiệm (xác suất ban đầu trước khi có dữ liệu mới)
- **likelihood**: xác suất quan sát được dữ liệu nếu giả thuyết đúng
- **evidence**: xác suất quan sát được dữ liệu (dùng để chuẩn hóa)

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

## Tải tập dữ liệu

In [3]:
df = pd.read_csv('/content/drive/MyDrive/Data_Analysis/DA06/mushrooms.csv')

df.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


Ở đây, từ bộ dữ liệu ta có thể thấy rằng:
- Tất cả các đặc trưng (features) đều là dữ liệu phân loại (categorical).
- Các giá trị của các đặc trưng này cần phải được mã hóa (encode) thành dạng số (numeric values) để sử dụng cho bộ phân loại (classifier).

## Mã hóa các đặc trưng thành dữ liệu số

Chúng ta sẽ sử dụng LabelEncoder cho nhiệm vụ này, công cụ này sẽ chuyển đổi các giá trị phân loại thành giá trị số thứ tự (ordinal values) cho tất cả các đặc trưng dạng phân loại.

Mặc dù đây không phải là phương pháp tối ưu nhất, nhưng cách làm này rất đơn giản và dễ áp dụng.

In [4]:
encoder = LabelEncoder()

# Apply the encoder to each of the columns
df = df.apply(encoder.fit_transform)

df.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,1,5,2,4,1,6,1,0,1,4,...,2,7,7,0,2,1,4,2,3,5
1,0,5,2,9,1,0,1,0,0,4,...,2,7,7,0,2,1,4,3,2,1
2,0,0,2,8,1,3,1,0,0,5,...,2,7,7,0,2,1,4,3,2,3
3,1,5,3,8,1,6,1,0,1,5,...,2,7,7,0,2,1,4,2,3,5
4,0,5,2,3,0,5,1,1,0,4,...,2,7,7,0,2,1,0,3,0,1


Như chúng ta có thể thấy, giá trị cho mỗi cột dao động từ 0 đến số lượng danh mục cho đặc điểm đó. Tiếp theo, chúng ta sẽ định nghĩa các hàm cho xác suất trước và xác suất sau để tính xác suất sau và so sánh với từng biến mục tiêu để xem điểm dữ liệu nào phù hợp với điểm nào.

## Chia bộ dữ liệu thành hai phần: train và test

In [5]:
# Seperating our target and features
X = df.drop(columns = ['class'])
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3)

print("X_train = ", X_train.shape)
print("y_train = ", y_train.shape)
print("X_test = ", X_test.shape)
print("y_test = ", y_test.shape)

X_train =  (5686, 22)
y_train =  (5686,)
X_test =  (2438, 22)
y_test =  (2438,)


## Xây dựng bộ phân loại của chúng ta

Cuối cùng, đây là lúc chúng ta bắt đầu xây dựng bộ phân loại Naive Bayes từng bước một.
Như đã thấy trước đó, để phân loại bất kỳ điểm dữ liệu nào trong Naive Bayes Classifier, ta cần hai yếu tố chính:
- Xác suất có điều kiện (likelihood)
- Xác suất tiên nghiệm (prior probability)

Sau đó, ta sẽ so sánh và gán nhãn (assign class) cho điểm dữ liệu dựa trên lớp có xác suất hậu nghiệm (posterior probability) cao hơn.

In [6]:
def prior(y_train, label):

    total_points = y_train.shape[0]
    class_points = np.sum(y_train == label)

    return class_points/float(total_points)

In [9]:
def cond_prob(X_train, y_train, feat_col, feat_val, label):
    """
    In this function, we will calculate the conditional probability which will be used to calculate likelihood.
    The value it returns is of the form
        P(x_i | y = C)
    which is the probability of the current feature (given by feat_col x_i) having the current value (given by feat_val)
    given that it belongs to the target class C

    Effectively, it reduces to the form
        all points belongig to class C which have the given value for the feature column / all points belonging to class C
    """

    # Getting all the
    X_filtered = X_train[y_train == label]

    numerator = np.sum(X_filtered[feat_col] == feat_val)
    denominator = np.sum(y_train == label)

    return numerator/float(denominator)

In [10]:
## Now time to calculate the posterior probability and make predictions

def predict(X_train, y_train, xtest):

    # Get the number of target classes
    classes = np.unique(y_train)

    # All the features for our dataset
    features = [x for x in X_train.columns]


    # Compute posterior probabilites for each class
    post_probs = []

    for label in classes:

        # Since, posterior = prior * likelihood
        # We'll calculate likelihood by calculating the product of the conditional probabilities for each of the features

        likelihood = 1.0

        for f in features:
            cond = cond_prob(X_train, y_train, f, xtest[f], label)
            likelihood *= cond

        prior_prob = prior(y_train, label)

        posterior = prior_prob * likelihood

        post_probs.append(posterior)

    # Return the label for which the posterior probability was the maximum
    prediction = np.argmax(post_probs)

    return prediction

## Kiểm tra bộ phân loại


In [11]:
# First, let's check on a random example

rand_example = 6

output = predict(X_train, y_train, X_test.iloc[rand_example])

print("Naive Bayes Classifier predicts ", output)
print("Current Answer ", y_test.iloc[rand_example])

Naive Bayes Classifier predicts  0
Current Answer  0


In [12]:
## Now, we'll check the results on each of the test data point and calculate
## an accuracy-based score for our classifier

def accuracy_score(X_train, y_train, xtest, ytest):

    preds = []

    for i in range(xtest.shape[0]):
        pred_label = predict(X_train, y_train, xtest.iloc[i])
        preds.append(pred_label)

    preds = np.array(preds)

    accuracy = np.sum(preds == ytest)/ytest.shape[0]

    return accuracy

- Một điểm dữ liệu ngẫu nhiên (ở đây là dòng thứ 6 trong tập kiểm tra) được chọn ra để mô hình dự đoán.
- Kết quả hiển thị cho thấy mô hình dự đoán giá trị đầu ra (output) trùng khớp với nhãn thực tế (current answer).

Điều này chứng tỏ mô hình đã học tốt và có khả năng dự đoán chính xác trên một mẫu cụ thể.

Đánh giá toàn bộ tập kiểm tra

- Một hàm accuracy_score() được xây dựng thủ công để kiểm tra tỷ lệ chính xác (Accuracy) của mô hình.
- Hàm này dự đoán từng điểm trong tập kiểm tra, sau đó so sánh với nhãn thật để tính tổng số dự đoán đúng.
- Mô hình đạt độ chính xác 99.75% (≈ 0.9975).

Đây là một kết quả rất cao, cho thấy mô hình Naive Bayes được xây dựng thủ công (from scratch) hoạt động hiệu quả và ổn định.

Sai số rất nhỏ có thể đến từ một vài điểm dữ liệu khó phân biệt hoặc có sự chồng lấn giữa các lớp.

In [13]:
print("Accuracy Score for our classifier == ", accuracy_score(X_train, y_train, X_test, y_test))

Accuracy Score for our classifier ==  0.9975389663658737
